# Layer 1: Crypto10K FinBERT Scoring and Adversarial Mutation

This notebook runs the Layer 1 pipeline in Colab: clean BTC/ETH tweets, score tweets with FinBERT, aggregate 5-minute EDA windows, generate adversarial injection rows, score synthetic rows, and save mutated datasets to Google Drive.

In [ ]:
!pip -q install transformers torch pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone the GitHub Repo

If the repo is public, this cell works without a token. If the repo is private, add a Colab Secret named `GITHUB_TOKEN` with read access to the repo before running this cell.

In [ ]:
from pathlib import Path
import os
import shutil
import stat
import subprocess
import sys

GITHUB_REPO = 'https://github.com/changju784/crypto-drift-guard.git'
GITHUB_BRANCH = 'layer1'  # Change to 'main' after this work is merged.
CLONE_DIR = Path('/content/crypto-drift-guard')

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')

env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
if GITHUB_TOKEN:
    askpass = Path('/content/git_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo oauth2 ;;\n'
        '  *) printf "%s" "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
    env['GIT_ASKPASS'] = str(askpass)
    env['GITHUB_TOKEN'] = GITHUB_TOKEN

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

cmd = ['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO, str(CLONE_DIR)]
subprocess.run(cmd, check=True, env=env)

sys.path.insert(0, str(CLONE_DIR / 'src'))
print('Cloned repo:', CLONE_DIR)
print('Using branch:', GITHUB_BRANCH)

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Drive is used for generated datasets so Colab outputs persist.
DRIVE_BASE = Path('/content/drive/MyDrive/crypto-drift-guard')
REPO_ROOT = CLONE_DIR
DATA_DIR = DRIVE_BASE / 'data'

for path in [
    DATA_DIR / 'raw', DATA_DIR / 'processed', DATA_DIR / 'scored',
    DATA_DIR / 'mutated' / 'sentiment', DATA_DIR / 'mutated' / 'temporal',
    DATA_DIR / 'mutated' / 'adversarial', DATA_DIR / 'reports'
]:
    path.mkdir(parents=True, exist_ok=True)

repo_src = str(REPO_ROOT / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

RAW_FILENAME = 'crypto_10k_tweets_(2021_2022Nov).csv'
RAW_CSV = DATA_DIR / 'raw' / RAW_FILENAME
if not RAW_CSV.exists():
    RAW_CSV = REPO_ROOT / 'data' / 'raw' / RAW_FILENAME

print('Repo root:', REPO_ROOT)
print('Drive data dir:', DATA_DIR)
print('Raw CSV:', RAW_CSV)

In [ ]:
from data.tweet_ingestion import clean_crypto10k
from data.finbert_scoring import add_finbert_scores, load_finbert
from data.window_features import aggregate_window_features, select_attack_target_windows
from data.adversarial_injection import generate_adversarial_tweets, merge_scored_synthetic

clean_df = clean_crypto10k(RAW_CSV)
clean_path = DATA_DIR / 'processed' / 'crypto10k_btc_eth_clean.csv'
clean_df.to_csv(clean_path, index=False)
print('Cleaned rows:', len(clean_df))
print(clean_df['cryptocurrency'].value_counts())
print('Saved:', clean_path)

In [ ]:
tokenizer, model, device = load_finbert()
batch_size = 64 if device == 'cuda' else 16
print('Device:', device, 'batch_size:', batch_size)

In [ ]:
scored_path = DATA_DIR / 'scored' / 'crypto10k_btc_eth_finbert_scored.csv'

if scored_path.exists():
    scored_df = pd.read_csv(scored_path, parse_dates=['timestamp'])
    print('Loaded cached scored dataset:', scored_path, len(scored_df))
else:
    scored_df = add_finbert_scores(
        clean_df,
        tokenizer,
        model,
        device=device,
        batch_size=batch_size,
        max_length=256,
    )
    scored_df.to_csv(scored_path, index=False)
    print('Saved:', scored_path)

print(scored_df[['cryptocurrency', 'finbert_label', 'sentiment_score']].head())
print(scored_df['finbert_label'].value_counts())

In [ ]:
window_df = aggregate_window_features(scored_df, freq='5min')
window_path = DATA_DIR / 'processed' / 'crypto10k_window_baseline_5min.csv'
window_df.to_csv(window_path, index=False)

eda_path = DATA_DIR / 'reports' / 'layer1_eda_summary.csv'
window_df.groupby(['cryptocurrency', 'regime']).size().reset_index(name='window_count').to_csv(eda_path, index=False)

print('Saved windows:', window_path)
print('Saved EDA summary:', eda_path)
display(window_df.groupby(['cryptocurrency', 'regime']).size().reset_index(name='window_count'))
display(window_df.sort_values(['cryptocurrency', 'window_start']).head(10))

In [ ]:
target_tables = []
for attack_type in ['positive_pump', 'negative_fud', 'conflict_balance']:
    targets = select_attack_target_windows(window_df, attack_type, max_windows=10)
    targets = targets.assign(attack_type=attack_type)
    target_tables.append(targets)

attack_targets = pd.concat(target_tables, ignore_index=True) if target_tables else pd.DataFrame()
target_path = DATA_DIR / 'reports' / 'layer1_attack_target_windows.csv'
attack_targets.to_csv(target_path, index=False)
print('Saved target windows:', target_path)
display(attack_targets[['attack_type', 'window_id', 'cryptocurrency', 'regime', 'tweet_count', 'positive_ratio', 'negative_ratio', 'mean_sentiment_score']])

In [ ]:
ATTACK_CONFIGS = [
    ('positive_pump', 0.70, 'crypto10k_positive_pump_r70.csv'),
    ('negative_fud', 0.70, 'crypto10k_negative_fud_r70.csv'),
    ('conflict_balance', 0.50, 'crypto10k_conflict_balance_r50.csv'),
]

for attack_type, target_ratio, filename in ATTACK_CONFIGS:
    synthetic = generate_adversarial_tweets(
        window_df,
        attack_type=attack_type,
        target_ratio=target_ratio,
        max_windows=10,
        seed=527,
    )
    unscored_path = DATA_DIR / 'mutated' / 'adversarial' / filename.replace('.csv', '_synthetic_unscored.csv')
    synthetic.to_csv(unscored_path, index=False)
    print('\nAttack:', attack_type, 'synthetic rows:', len(synthetic), 'saved:', unscored_path)

    if synthetic.empty:
        continue

    synthetic_scored = add_finbert_scores(
        synthetic,
        tokenizer,
        model,
        device=device,
        batch_size=batch_size,
        max_length=256,
    )
    mutated_scored = merge_scored_synthetic(scored_df, synthetic_scored)
    out_path = DATA_DIR / 'mutated' / 'adversarial' / filename
    mutated_scored.to_csv(out_path, index=False)

    mutated_windows = aggregate_window_features(mutated_scored, freq='5min')
    windows_out = DATA_DIR / 'mutated' / 'adversarial' / filename.replace('.csv', '_windows_5min.csv')
    mutated_windows.to_csv(windows_out, index=False)
    print('Saved scored mutation:', out_path)
    print('Saved mutation windows:', windows_out)